# Session 4b · Serve the model: a REST API
In `Session04_Model.ipynb` we saved the model as `models/mpg_model.joblib`. Now **other programs** shall use it: a web app, a phone app, a C program on a microcontroller gateway. They don't have Python or scikit-learn. They only speak **HTTP + JSON**.

```
 client (any language)                      server (Python, our model)
 POST /predict  {"weight": 2400, ...}  -->  joblib.load() once, model.predict()
                {"mpg": 33.9, ...}     <--
```
Run `Session04_Model.ipynb` first: it creates the model file.

In [1]:
import json
import subprocess
import sys
import time
from pathlib import Path

import joblib
import pandas as pd
import requests

MODEL = Path("models/mpg_model.joblib")
print("model file found:", MODEL.exists())

model file found: True


### 1 · Recap: load and predict, inside Python

In [2]:
model = joblib.load(MODEL)
meta = json.loads(Path("models/mpg_model.json").read_text())

car = {"cylinders": 4, "displacement": 120, "horsepower": 90, "weight": 2400,
       "acceleration": 16, "model_year": 80, "origin": "japan"}
print(f"{model.predict(pd.DataFrame([car]))[0]:.1f} mpg")

33.9 mpg


This only works where Python, scikit-learn **in the right version** and the model file are available. A REST API puts all of that on one server.

### 2 · REST in five lines
| | means |
|---|---|
| **URL / endpoint** | an address on the server, e.g. `/predict` |
| **GET** | read something, no body (`/health`: "are you alive?") |
| **POST** | send data in the body, get a result (`/predict`) |
| **JSON** | the data format of the body: `{"weight": 2400}` |
| **status code** | 200 = OK, 422 = your input is invalid, 500 = server error |

### 3 · The server: `api.py`
We use **FastAPI**. The class `Car` describes what a valid request looks like: FastAPI checks every request against it *before* our model sees it.

In [3]:
%%writefile api.py
"""REST API for the Auto MPG model. Start with:  python -m uvicorn api:app --port 8000"""
import json
from pathlib import Path
from typing import Literal

import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

MODEL_DIR = Path(__file__).parent / "models"
model = joblib.load(MODEL_DIR / "mpg_model.joblib")                     # load ONCE at start, not per request
meta = json.loads((MODEL_DIR / "mpg_model.json").read_text())

app = FastAPI(title="Auto MPG model")


class Car(BaseModel):                                                    # the input contract
    cylinders: int = Field(ge=3, le=12)
    displacement: float = Field(gt=0, description="cubic inches")
    horsepower: float | None = Field(default=None, gt=0)                 # may be missing: the pipeline imputes it
    weight: float = Field(gt=0, description="lb")
    acceleration: float = Field(gt=0, description="seconds 0-60 mph")
    model_year: int = Field(ge=70, le=82)
    origin: Literal["usa", "europe", "japan"]


class Prediction(BaseModel):                                             # the output contract
    mpg: float
    expected_error_mpg: float


@app.get("/health")
def health():
    return {"status": "ok", "trained_at": meta["trained_at"], "test_mae": meta["test_mae"]}


@app.post("/predict")
def predict(car: Car) -> Prediction:
    X = pd.DataFrame([car.model_dump()])                                 # JSON -> one-row DataFrame, like in training
    return Prediction(mpg=round(float(model.predict(X)[0]), 1),
                      expected_error_mpg=meta["test_mae"])

Writing api.py


### 4 · Start the server
Normally you run this in a terminal: `python -m uvicorn api:app --port 8000`. From the notebook we start it as a separate process, so the notebook stays free for the client.

In [4]:
PORT = 8000
URL = f"http://127.0.0.1:{PORT}"

server = subprocess.Popen([sys.executable, "-m", "uvicorn", "api:app", "--port", str(PORT)],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(50):                      # wait until the server answers (loading the model takes a moment)
    try:
        requests.get(f"{URL}/health", timeout=0.5)
        break
    except requests.ConnectionError:
        time.sleep(0.2)
print("server running at", URL)

server running at http://127.0.0.1:8000


### 5 · The client: from here on, no scikit-learn needed

In [5]:
r = requests.get(f"{URL}/health")
print(r.status_code)
r.json()

200


{'status': 'ok', 'trained_at': '2026-11-01 16:49', 'test_mae': 1.599}

In [6]:
r = requests.post(f"{URL}/predict", json=car)     # requests turns the dict into JSON
print(r.status_code)
r.json()

200


{'mpg': 33.9, 'expected_error_mpg': 1.599}

Same number as in section 1, but this time the prediction came over the network.
### 6 · Invalid input: the API protects the model

In [7]:
bad_car = {**car, "weight": "heavy", "origin": "mars"}
r = requests.post(f"{URL}/predict", json=bad_car)
print(r.status_code)
[(e["loc"][-1], e["msg"]) for e in r.json()["detail"]]

422


[('weight',
  'Input should be a valid number, unable to parse string as a number'),
 ('origin', "Input should be 'usa', 'europe' or 'japan'")]

Status **422**: the request never reached the model. Without this check, `"heavy"` would crash the pipeline (500) or, worse, a wrong number would give a silent wrong prediction.

In [8]:
# horsepower is optional: the pipeline's imputer fills it with the training median
r = requests.post(f"{URL}/predict", json={k: v for k, v in car.items() if k != "horsepower"})
r.json()

{'mpg': 34.0, 'expected_error_mpg': 1.599}

### 7 · Any language can call it
The same request with `curl` from a terminal (works on macOS, Linux and Windows 10+):

In [9]:
!curl -s -X POST {URL}/predict -H "Content-Type: application/json" -d '{json.dumps(car)}'

{"mpg":33.9,"expected_error_mpg":1.599}

FastAPI also generates an interactive documentation page: open **http://127.0.0.1:8000/docs** in the browser while the server runs. You can send test requests from there.
### 8 · How fast is it?

In [10]:
start = time.perf_counter()
for _ in range(100):
    requests.post(f"{URL}/predict", json=car)
print(f"{(time.perf_counter() - start) / 100 * 1000:.1f} ms per request")

9.2 ms per request


A few milliseconds, most of it the 300 trees of the random forest. Loading the model takes much longer: that's why `api.py` loads it **once at start**, not in every request.
### 9 · Stop the server

In [11]:
server.terminate()
server.wait()
print("server stopped")

server stopped


### Summary
| Step | Where |
|---|---|
| train, evaluate, `joblib.dump()` | `Session04_Model.ipynb` (runs rarely) |
| `joblib.load()` once, `predict()` per request | `api.py` (runs all the time) |
| send JSON, receive JSON | any client: Python `requests`, `curl`, Streamlit, a phone app |

**Stock App:** the Streamlit app can either load the model itself, or call such an API. With an API, several apps share one model, and updating the model means replacing one file on one server. Chapter 05 (Deployment) packs `api.py` + model into a Docker container.